# Model A
---

Load the preprocessed datasets, vocabulary, config and dataloaders using the following cell.

In [10]:
from utils import load_sets, load_vocab, load_config, get_dataloaders

train_set, val_set, test_set = load_sets()
vocab = load_vocab()
vocab_reversed = load_vocab(rev=True)
config = load_config()

train_loader, val_loader, test_loader = get_dataloaders(
    train_set, val_set, test_set,
    vocab,
    config['MAX_LENGTH'],
    config['BATCH_SIZE']
)

In [11]:
train_set.head()

,input,response
0,<S0> someone can help me?,<S1> to do what? <SEP> sry not yet on my way t...
1,<S0> can someone help me with my grub2 issue? ...,<S1> only if you tell us the issue :)
2,<S0> bonjour <SEP> g besoin d'aide,<S1> #ubuntu-fr pour francais
3,<S0> join /#london <SEP> whoops,<S1> bit of a mixup? ;)
4,<S0> which ppa did you add?,<S1> i found a y ppa admin tool and sorted it ...


In [12]:
config

{'MAX_LENGTH': 22,
 'MIN_TOKEN_FREQ': 5,
 'MIN_LENGTH': 4,
 'BATCH_SIZE': 64,
 'VOCAB_SIZE': 21953,
 'PAD_IDX': 0,
 'SOS_IDX': 1,
 'EOS_IDX': 2,
 'UNK_IDX': 3,
 'SEP_IDX': 4,
 'S0_IDX': 5,
 'S1_IDX': 6,
 'URL_IDX': 7,
 'IP_IDX': 8,
 'PATH_IDX': 9}

In [13]:
import torch

# Always use cuda if available (NVIDIA GPU), then try for apple silicon, otherwise just a plain old CPU (but training will be vastly slower)
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f'Using device: {torch.cuda.get_device_name(0)}')
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print('Using device: MPS (Apple Silicon)')
else:
    device = torch.device("cpu")
    print('Using device: CPU')

Using device: Orin


Quick plan:

1. Define the Encoder  — GRU that reads input, returns ~~all hidden states~~ + final hidden state
2. Define the Decoder  — GRU that generates output token by token using final encoder hidden state
3. Define the Seq2Seq  — wrapper that connects encoder and decoder, handles teacher forcing
4. Training loop       — with validation loss monitoring
5. Inference function  — greedy decoding for generating responses
6. Evaluation          — BLEU score on test set + manual examples

## Encoder

---

The encoder will read the input sequences we've prepared, and will output a single final hidden state to initialise the decoder. So, the architecture is actually quite simple:
- An embedding layer (dataset has low GloVe coverage, so we can train embeddings from scratch).
- A stack of Gated Recurrent Units (combats vanishing gradients, fewer parameters to train than an LSTM, with comparable performance).
- Dropout as a regularisation technique to combat overfitting and improve generalisation (only applied during training). We'll apply dropout between the embedding layer, and between the layers in the GRU.

Defining a sequential model in PyTorch follows the usual pattern:
- Extend the [`nn.Module`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html) base class.
- Initialise your layers in the constructor.
- Override the `forward` method, explicitly feeding X inputs through your layers.

Please read the [GRU](https://docs.pytorch.org/docs/stable/generated/torch.nn.GRU.html) documentation, it explains a lot! :)

In [14]:
from encoder import Encoder

## Decoder

---

The decoder must first initialise its hidden state with the final output hidden state of the encoder. Then, at each timestep (**a forward pass through the network processing one token**):
1. Obtain the embedding for the previous token
2. Pass it through the GRU
3. Project the output to the vocabulary size to predict the next token (using a simple fully connected linear layer)

Outside of this, its layer initialisation is very similar to the encoder. 

Note we're also training separate embeddings in the decoder rather using shared embeddings between the encoder and decoder (called _weight tying_ -- possible note in the report). This is to keep the embeddings specialised as the encoder and decoder objectives are fundamentally different: the encoder must encode meaning into the hidden state, the decoder must utilise this state to predict the next token given the previous one. Therefore, allowing both to represent tokens in a way that optimises their objectives is the intent.

In [15]:
from decoder import Decoder

## Seq2Seq Wrapper

---

This is the wrapper model that will be built from the encoder and decoder. It will:
1. Pass the full input sequence through the encoder to produce a final hidden state
2. Use this final hidden state to initialise the decoder
3. Loop through each decoding timestep and feed each token in one at a time
4. Apply **teacher forcing** during training only (feeding the ground truth labels in at each timestep to improve stability and reduce _exposure bias_I)

The variables passed into the `forward` method are those we prepared at the data preparation stage:
- `encoder_input`: the encoded input sequence (for the encoder)
- `decoder_input`: the encoded response with SOS prepended

In [16]:
from seq2Seq import Seq2Seq

## Training Loop

---

This is the main training loop, where we'll use cross entropy loss, Adam optimiser, and gradient clipping to manage exploding gradients (this is common in BPTT).

We can use a `max_norm=1.0` to scale the gradient vector down if its norm exceeds 1 to help keep the weight updates stable.

First things first: let's define the training and model hyperparameters, and instantiate the model.

In [17]:
# Training Parameters
EPOCHS = 80
CLIP_MAX_NORM = 50.0
TEACHER_FORCING_PROBA = 0.5
ENCODER_LEARNING_RATE=1e-3
DECODER_LEARNING_RATE=ENCODER_LEARNING_RATE * 2

# Model Hyperparameters
ENCODER_EMBED_DIM = 128
ENCODER_HIDDEN_DIM = 256
ENCODER_NUM_LAYERS = 2
ENCODER_DROPOUT_PROBA = 0.3

DECODER_EMBED_DIM = 128
DECODER_HIDDEN_DIM = 256
DECODER_NUM_LAYERS = 2
DECODER_DROPOUT_PROBA = 0.3

model_a = Seq2Seq(
    encoder=Encoder(
        vocab_size=config['VOCAB_SIZE'], 
        embed_dim=ENCODER_EMBED_DIM, 
        padding_idx=config['PAD_IDX'], 
        hidden_dim=ENCODER_HIDDEN_DIM,
        num_layers=ENCODER_NUM_LAYERS,
        dropout_proba=ENCODER_DROPOUT_PROBA
    ),
    decoder = Decoder(
        vocab_size=config['VOCAB_SIZE'],
        embed_dim=DECODER_EMBED_DIM,
        padding_idx=config['PAD_IDX'],
        hidden_dim=DECODER_HIDDEN_DIM,
        num_layers=DECODER_NUM_LAYERS,
        dropout_proba=DECODER_DROPOUT_PROBA
    ),
    device=device
).to(device) # Move model to device

In [18]:
#!mkdir -p models/model_a
from pathlib import Path

Path("models/model_a").mkdir(parents=True, exist_ok=True)

In [ ]:
from train import train

train(
    model=model_a,
    train_loader=train_loader,
    val_loader=val_loader,
    vocab_reversed=vocab_reversed,
    config=config,
    device=device,
    checkpoint_dir='models/model_a',
    hyperparams={
        'EPOCHS': EPOCHS,
        'CLIP_MAX_NORM': CLIP_MAX_NORM,
        'TEACHER_FORCING_PROBA': TEACHER_FORCING_PROBA,
        'ENCODER_LEARNING_RATE': ENCODER_LEARNING_RATE,
        'DECODER_LEARNING_RATE': DECODER_LEARNING_RATE
    }
)

No checkpoints, training model from scratch...
